In [3]:
import Pkg; Pkg.activate(@__DIR__); Pkg.instantiate()
using LinearAlgebra
using ForwardDiff
using PyPlot


  Activating project at `~/workspace/myProject/julia_ws`


In [4]:
function ToCross(vec3)
    return [0       -vec3[3]    vec3[2];
            vec3[3]     0      -vec3[1];
           -vec3[2]  vec3[1]     0      ]
end


ToCross (generic function with 1 method)

In [5]:

function Exp(axis ,θ)
    rotm=Matrix(I, 3, 3)+sin(θ)*ToCross(axis)+(1-cos(θ))*ToCross(axis)^2
    return rotm
end


Exp (generic function with 1 method)

In [6]:

rotm_test=Exp([0 0 1],pi/2)
print(rotm_test)

[1.1102230246251565e-16 -1.0 0.0; 1.0 1.1102230246251565e-16 0.0; 0.0 0.0 1.0]

In [ ]:

function fkine(x)
    Tlist=[0 0 1 x[1];0 1 0 x[2];-1 0 0 x[3];0 0 0 1]
    return Tlist
end

In [ ]:



Q = Diagonal([0.5; 1])
function f(x)
    return 0.5*(x-[1; 0])'*Q*(x-[1; 0])
end


In [3]:

function ∇f(x)
    return Q*(x-[1; 0])
end


∇f (generic function with 1 method)

In [4]:

function ∇2f(x)
    return Q
end


∇2f (generic function with 1 method)

In [5]:

function c(x)
    return x[1]^2 + 2*x[1] - x[2]
end


c (generic function with 1 method)

In [6]:
function ∂c(x)
    return [2*x[1]+2 -1]
end


∂c (generic function with 1 method)

In [ ]:
function plot_landscape()
    Nsamp = 20
    Xsamp = kron(ones(Nsamp),LinRange(-4,4,Nsamp)')
    Ysamp = kron(ones(Nsamp)',LinRange(-4,4,Nsamp))
    Zsamp = zeros(Nsamp,Nsamp)
    for j = 1:Nsamp
        for k = 1:Nsamp
            Zsamp[j,k] = f([Xsamp[j,k]; Ysamp[j,k]])
        end
    end
    contour(Xsamp,Ysamp,Zsamp)

    xc = LinRange(-3.2,1.2,Nsamp)
    plot(xc,xc.^2+2.0.*xc,"y")
end

plot_landscape()


In [ ]:


function newton_step(x0,λ0)
    H = ∇2f(x0) + ForwardDiff.jacobian(x -> ∂c(x)'*λ0, x0)
    C = ∂c(x0)
    Δz = [H C'; C 0]\[-∇f(x0)-C'*λ0; -c(x0)]
    Δx = Δz[1:2]
    Δλ = Δz[3]
    return x0+Δx, λ0+Δλ
end
xguess = [-3; 2]
λguess = [0.0]
plot_landscape()
plot(xguess[1], xguess[2], "rx")
xnew, λnew = newton_step(xguess[:,end],λguess[end])
xguess = [xguess xnew]
λguess = [λguess λnew]
plot_landscape()
plot(xguess[1,:], xguess[2,:], "rx")
H = ∇2f(xguess[:,end]) + ForwardDiff.jacobian(x -> ∂c(x)'*λguess[end], xguess[:,end])


In [ ]:

function gauss_newton_step(x0,λ0)
    H = ∇2f(x0)
    C = ∂c(x0)
    Δz = [H C'; C 0]\[-∇f(x0)-C'*λ0; -c(x0)]
    Δx = Δz[1:2]
    Δλ = Δz[3]
    return x0+Δx, λ0+Δλ
end
xguess = [-3; 2]
λguess = [0.0]
plot_landscape()
plot(xguess[1], xguess[2], "rx")
xnew, λnew = gauss_newton_step(xguess[:,end],λguess[end])
xguess = [xguess xnew]
λguess = [λguess λnew]
plot_landscape()
plot(xguess[1,:], xguess[2,:], "rx")
